<a href="https://colab.research.google.com/github/vikrampal12345/Resume_Analyser_Job_recommendation_Research/blob/main/03_Sentence_Transformer/03_Sentence_Transformer_Training_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate scikit-learn

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu128
0.26.0+cu128


In [ ]:
!pip install -U datasets transformers

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv("/content/ready_for_training_resume.csv")

print(df.shape)

df.head()

(2707, 17)


,Unnamed: 0,resume_text,skills,education,experience_years,projects,certifications,job_role_1,confidence_1,job_role_2,confidence_2,job_role_3,confidence_3,job_role_4,confidence_4,job_role_5,confidence_5
0,1,howard gerrard accountant work experience dyrp...,"['Accounting', 'Financial Accounting', 'VAT Re...",Bachelor of Commerce,12,[],[],accountant,100,financialaccountant,95,budgetanalyst,85,cashmanager,80,financialcontroller,70
1,2,olivia ogilvy accountant pacific ave los angel...,"['Financial Reporting', 'Accounting', 'Reconci...",MBA,7,[],['Chartered Financial Analyst (CFA)'],accountant,100,financialanalyst,90,accountingmanager,80,financemanager,70,financialcontroller,60
2,3,brooklyn ny stephenbeamjobs com linkedin steph...,"['QuickBooks', 'Taxjar', 'GAAP accounting prin...",MBA Accounting,9,[],['Certified Public Accountant (CPA)'],accountant,100,financialcontroller,90,accountingmanager,80,financemanager,70,senioraccountant,100
3,4,marie anderson sometown co mac somedomain com ...,"['QuickBooks', 'SQL', 'SAP', 'Oracle', 'GAAP',...",Bachelor of Science in Accounting,10,[{'name': 'Tech Startup Accounting System Impl...,[],accountant,100,financialanalyst,90,auditspecialist,85,taxaccountant,80,complianceofficer,75
4,5,annanovoresumecom anna gunther senior accounta...,"['Accounting', 'Financial Reporting', 'Cost Co...",Senior Accountant,10,[],['Certified Public Accountant'],accountant,100,financialaccountant,95,accountingmanager,85,financeanalyst,80,financialcontroller,70


In [ ]:
df = df.dropna(
    subset=[
        "resume_text",
        "job_role_1"
    ]
).reset_index(drop=True)

In [ ]:
job_columns = [
    "job_role_1",
    "job_role_2",
    "job_role_3",
    "job_role_4",
    "job_role_5"
]

df["labels"] = df[job_columns].values.tolist()

In [63]:
job_columns = [
    "job_role_1",
    "job_role_2",
    "job_role_3",
    "job_role_4",
    "job_role_5"
]

In [64]:
print(job_columns)

['job_role_1', 'job_role_2', 'job_role_3', 'job_role_4', 'job_role_5']


In [ ]:
df

In [ ]:
print(df.shape)

(2707, 18)


In [ ]:
X = df["resume_text"]

In [ ]:
X

,resume_text
0,howard gerrard accountant work experience dyrp...
1,olivia ogilvy accountant pacific ave los angel...
2,brooklyn ny stephenbeamjobs com linkedin steph...
3,marie anderson sometown co mac somedomain com ...
4,annanovoresumecom anna gunther senior accounta...
...,...
2702,web designer resume creative with 10 years of ...
2703,stephanie smith senior web developer innovativ...
2704,first last freelance web developer new york ci...
2705,karen philips web designer info profile addres...


In [ ]:
df['resume_text'].shape

(2707,)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
resume_embeddings = model.encode(
    df["resume_text"].tolist(),
    batch_size=32,    # it convert 32 resume in one batch 1-32 and 32 - 64 so on..
    show_progress_bar=True,
    convert_to_numpy=True   # numpy array convert
)

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

In [ ]:
print(resume_embeddings.shape)  # miniLM works on the 384 dimension

(2707, 384)


In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np

dimension = resume_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)   # L2 works on the euclidean distance ,
# index = faiss.IndexFlatIP(dimension)   # IP works on the cosine similarity


index.add(
    np.array(resume_embeddings).astype("float32")  # add the index and the vector
)

In [ ]:
print(index.ntotal)

2707


In [ ]:
query = model.encode(    # it give first index resume, which is for testing,
    [df.iloc[0]["resume_text"]],  # here user put the resume
    convert_to_numpy=True
).astype("float32")

In [ ]:
distance, indices = index.search(
    query,
    k=10        # it find 10 similar resume as compare to 0 index resume
)

In [61]:
distance

array([[6.3451517e-13, 6.3451517e-13, 4.2671826e-01, 4.6591303e-01,
        5.0330931e-01, 5.0330931e-01, 5.4994828e-01, 5.7608479e-01,
        5.8558130e-01, 5.9296334e-01]], dtype=float32)

In [ ]:
print(indices)  # it return 10 similar resume

[[   0 2459   13   30   25 2458 2460   29  493   15]]


In [51]:
indices[0]

array([   0, 2459,   13,   30,   25, 2458, 2460,   29,  493,   15])

In [58]:
recommendations = []

for idx in indices[0]:

    recommendations.extend([
        df.iloc[idx]["job_role_1"],  # resume 0 index, extract job_role_1
        df.iloc[idx]["job_role_2"],
        df.iloc[idx]["job_role_3"],
        df.iloc[idx]["job_role_4"],
        df.iloc[idx]["job_role_5"]
    ])

In [59]:
df.iloc[0]['job_role_1']

'accountant'

In [60]:
print(recommendations)

['accountant', 'financialaccountant', 'budgetanalyst', 'cashmanager', 'financialcontroller', 'accountant', 'financialaccountant', 'budgetanalyst', 'financialcontroller', 'accountingmanager', 'accountant', 'financialanalyst', 'accountingmanager', 'financemanager', 'accountingspecialist', 'accountant', 'financialcontroller', 'businessanalyst', 'complianceofficer', 'accountingmanager', 'accountant', 'financialanalyst', 'accountingmanager', 'controller', 'financemanager', 'accountant', 'financialanalyst', 'accountingmanager', 'controller', 'financemanager', 'accountant', 'financialaccountant', 'accountingmanager', 'financeanalyst', 'accountingsupervisor', 'accountant', 'financialcontroller', 'accountingmanager', 'financemanager', 'financialanalyst', 'financialanalyst', 'accountant', 'financemanager', 'financialconsultant', 'accountingmanager', 'accountant', 'financialanalyst', 'accountingmanager', 'financemanager', 'accountingspecialist']


In [ ]:
# from collections import Counter

# top_jobs = Counter(recommendations)

# print(top_jobs.most_common(5))

In [67]:
distance[0][1]

np.float32(6.3451517e-13)

In [ ]:
from collections import defaultdict
import numpy as np

job_scores = defaultdict(float)

for rank, idx in enumerate(indices[0]):

    # Convert L2 distance to similarity
    similarity = 1 / (1 + distance[0][rank])

    for col in job_columns:

        job = df.iloc[idx][col]

        if pd.notna(job):
            job_scores[job] += similarity

top_jobs = sorted(
    job_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(top_jobs[:5])

[('accountant', np.float32(7.251587)), ('accountingmanager', np.float32(6.251587)), ('financialanalyst', np.float32(3.9242353)), ('financemanager', np.float32(3.9242353)), ('financialcontroller', np.float32(3.3166523))]


In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

In [ ]:
total_hit = 0

total_precision = 0

total_recall = 0

total_resume = len(df)

In [ ]:
for i in range(total_resume):

    # Query Resume
    query = model.encode(
        [df.iloc[i]["resume_text"]],
        convert_to_numpy=True
    ).astype("float32")

    # Search Top 20 Similar Resumes
    distance, indices = index.search(query, k=20)

    # Score Every Job Role
    job_scores = defaultdict(float)

    for rank, idx in enumerate(indices[0]):

        # Skip Same Resume
        if idx == i:
            continue

        similarity = 1 / (1 + distance[0][rank])

        for col in job_columns:

            job = df.iloc[idx][col]

            if pd.notna(job):
                job_scores[job] += similarity

    # Top 5 Recommendation
    predicted_jobs = [
        job
        for job, score in sorted(
            job_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:5]
    ]

    # Actual Jobs
    actual_jobs = set()

    for col in job_columns:

        value = df.iloc[i][col]

        if pd.notna(value):
            actual_jobs.add(value)

    predicted_jobs = set(predicted_jobs)

    # Common Jobs
    correct = len(
        actual_jobs.intersection(predicted_jobs)
    )

    # Hit@5
    if correct > 0:
        total_hit += 1

    # Precision@5
    total_precision += correct / 5

    # Recall@5
    total_recall += correct / len(actual_jobs)

In [ ]:
hit5 = total_hit / total_resume

precision5 = total_precision / total_resume

recall5 = total_recall / total_resume

print("=" * 50)

print(f"Hit@5        : {hit5:.4f}")

print(f"Precision@5  : {precision5:.4f}")

print(f"Recall@5     : {recall5:.4f}")

print("=" * 50)

Hit@5        : 0.8633
Precision@5  : 0.3990
Recall@5     : 0.3995


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 2165
Test : 542


In [ ]:
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
train_embeddings = model.encode(
    train_df["resume_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

dimension = train_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(train_embeddings).astype("float32")
)

print(index.ntotal)

2165


In [ ]:
from collections import defaultdict

total_hit = 0

total_precision = 0

total_recall = 0

total_resume = len(test_df)

In [ ]:
for i in range(total_resume):

    query = model.encode(
        [test_df.iloc[i]["resume_text"]],
        convert_to_numpy=True
    ).astype("float32")

    distance, indices = index.search(query, k=20)

    job_scores = defaultdict(float)

    for rank, idx in enumerate(indices[0]):

        similarity = 1 / (1 + distance[0][rank])

        for col in job_columns:

            job = train_df.iloc[idx][col]

            if pd.notna(job):
                job_scores[job] += similarity

    predicted_jobs = [
        job
        for job, score in sorted(
            job_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:5]
    ]

    predicted_jobs = set(predicted_jobs)

    actual_jobs = set()

    for col in job_columns:

        value = test_df.iloc[i][col]

        if pd.notna(value):
            actual_jobs.add(value)

    correct = len(
        predicted_jobs.intersection(actual_jobs)
    )

    if correct > 0:
        total_hit += 1

    total_precision += correct / 5

    total_recall += correct / len(actual_jobs)

In [ ]:
hit5 = total_hit / total_resume

precision5 = total_precision / total_resume

recall5 = total_recall / total_resume

print("=" * 50)

print(f"Hit@5        : {hit5:.4f}")

print(f"Precision@5  : {precision5:.4f}")

print(f"Recall@5     : {recall5:.4f}")

print("=" * 50)

Hit@5        : 0.8708
Precision@5  : 0.4151
Recall@5     : 0.4156


In [ ]:
for i in range(5):

    query = model.encode(
        [test_df.iloc[i]["resume_text"]],
        convert_to_numpy=True
    ).astype("float32")

    distance, indices = index.search(query, k=20)

    job_scores = defaultdict(float)

    for rank, idx in enumerate(indices[0]):

        similarity = 1 / (1 + distance[0][rank])

        for col in job_columns:

            job = train_df.iloc[idx][col]

            if pd.notna(job):
                job_scores[job] += similarity

    predicted = [
        job
        for job, score in sorted(
            job_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:5]
    ]

    actual = [
        test_df.iloc[i][col]
        for col in job_columns
        if pd.notna(test_df.iloc[i][col])
    ]

    print("=" * 70)
    print("Actual    :", actual)
    print("Predicted :", predicted)

Actual    : ['accountant', 'manager', 'financialcontroller', 'accountingmanager', 'payrolladministrator']
Predicted : ['accountant', 'accountingmanager', 'financialanalyst', 'financemanager', 'financialcontroller']
Actual    : ['electricalengineer', 'controlsystemsengineer', 'powersystemsengineer', 'processcontrolengineer', 'mechatronicsengineer']
Predicted : ['mechanicalengineer', 'projectmanager', 'softwareengineer', 'electricalengineer', 'technicalsupportengineer']
Actual    : ['casemanager', 'humanresourcesmanager', 'communityoutreachspecialist', 'programdirector', 'socialservicesadministrator']
Predicted : ['projectmanager', 'humanresourcesmanager', 'operationsmanager', 'specialeducationteacher', 'studentservicescoordinator']
Actual    : ['databaseadministrator', 'softwareengineer', 'dataanalyst', 'backenddeveloper', 'systemsadministrator']
Predicted : ['databaseadministrator', 'dataanalyst', 'dataengineer', 'etldeveloper', 'businessanalyst']
Actual    : ['businessanalyst', 'sapco

In [ ]:
faiss.write_index(index, "resume_index.faiss")

In [ ]:
np.save("resume_embeddings.npy", train_embeddings)

In [ ]:
train_df.to_csv("resume_database.csv", index=False)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

model.save("models/minilm_model")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import os

print(os.listdir("models/minilm_model"))

['tokenizer_config.json', 'model.safetensors', '2_Normalize', 'sentence_bert_config.json', 'tokenizer.json', 'config.json', 'README.md', 'config_sentence_transformers.json', '1_Pooling', 'modules.json']


In [ ]:
!zip -r minilm_model.zip models/minilm_model

  adding: models/minilm_model/ (stored 0%)
  adding: models/minilm_model/tokenizer_config.json (deflated 51%)
  adding: models/minilm_model/model.safetensors (deflated 9%)
  adding: models/minilm_model/2_Normalize/ (stored 0%)
  adding: models/minilm_model/sentence_bert_config.json (deflated 43%)
  adding: models/minilm_model/tokenizer.json (deflated 71%)
  adding: models/minilm_model/config.json (deflated 52%)
  adding: models/minilm_model/README.md (deflated 64%)
  adding: models/minilm_model/config_sentence_transformers.json (deflated 40%)
  adding: models/minilm_model/1_Pooling/ (stored 0%)
  adding: models/minilm_model/1_Pooling/config.json (deflated 16%)
  adding: models/minilm_model/modules.json (deflated 64%)


In [ ]:
from google.colab import files

files.download("minilm_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>